# 🏀 KOBRA – NBA Game Prediction Model

Dieses Notebook teilt das Modell in einzelne Schritte auf, damit du jeden Schritt einzeln testen und anpassen kannst.

**Ablauf:**
1. Setup & Konfiguration
2. Spieldaten laden
3. Daten anschauen
4. Features berechnen (Winrate, Punkte)
5. Elo-Rating
6. Modell trainieren & testen
7. Heutige Spiele vorhersagen
8. Verletzungen & Impact
9. Finale Vorhersagen
10. Experimentieren (andere Modelle, Features, etc.)

## 1. Setup & Konfiguration
Hier werden alle Bibliotheken geladen und die API-Verbindung eingerichtet.

In [2]:
import os
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# === HIER DEINEN API KEY EINTRAGEN ===
API_KEY = "98ee804c-c05f-4eb3-88cd-3b7d42ae474e"

HEADERS = {"Authorization": API_KEY}
BASE_URL = "https://api.balldontlie.io/v1"

# Einstellungen – kannst du anpassen!
ELO_START = 1500
ELO_K = 20                    # Wie stark reagiert Elo auf ein Ergebnis
ELO_SEASON_REGRESSION = 0.25  # 25% Regression zum Mittelwert zwischen Saisons
FORM_WINDOW = 10              # Letzte X Spiele für Winrate/Punkte

print("✅ Setup fertig!")

✅ Setup fertig!


In [4]:
# Team-Name Mapping: ESPN Kurznamen → volle balldontlie-Namen

TEAM_NAME_MAP = {
    "Hawks": "Atlanta Hawks", "Celtics": "Boston Celtics",
    "Nets": "Brooklyn Nets", "Hornets": "Charlotte Hornets",
    "Bulls": "Chicago Bulls", "Cavaliers": "Cleveland Cavaliers",
    "Mavericks": "Dallas Mavericks", "Nuggets": "Denver Nuggets",
    "Pistons": "Detroit Pistons", "Warriors": "Golden State Warriors",
    "Rockets": "Houston Rockets", "Pacers": "Indiana Pacers",
    "Clippers": "LA Clippers", "Lakers": "Los Angeles Lakers",
    "Grizzlies": "Memphis Grizzlies", "Heat": "Miami Heat",
    "Bucks": "Milwaukee Bucks", "Timberwolves": "Minnesota Timberwolves",
    "Pelicans": "New Orleans Pelicans", "Knicks": "New York Knicks",
    "Thunder": "Oklahoma City Thunder", "Magic": "Orlando Magic",
    "76ers": "Philadelphia 76ers", "Suns": "Phoenix Suns",
    "Trail Blazers": "Portland Trail Blazers", "Kings": "Sacramento Kings",
    "Spurs": "San Antonio Spurs", "Raptors": "Toronto Raptors",
    "Jazz": "Utah Jazz", "Wizards": "Washington Wizards",
}

print(f"✅ {len(TEAM_NAME_MAP)} Teams gemappt")

✅ 30 Teams gemappt


## 2. Spieldaten laden
API-Funktionen und Daten von balldontlie holen. Das dauert ein paar Minuten wegen Rate Limits.

In [ ]:
def api_request(endpoint, params, max_retries=5):
    """Robuster API-Request mit Retry-Logik."""
    url = f"{BASE_URL}/{endpoint}"
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                wait = 60 * (attempt + 1)
                print(f"Rate limit. Warte {wait}s... (Versuch {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                print(f"HTTP {r.status_code}: {r.text[:200]}")
                time.sleep(10)
                continue
            return r.json()
        except Exception as e:
            print(f"Fehler: {e}. Warte 30s...")
            time.sleep(30)
    return None


def lade_spiele(saison):
    """Alle Spiele einer Saison paginiert laden."""
    spiele = []
    cursor = None
    while True:
        params = {"seasons[]": saison, "per_page": 100}
        if cursor:
            params["cursor"] = cursor
        data = api_request("games", params)
        if data is None:
            break
        spiele.extend(data["data"])
        cursor = data.get("meta", {}).get("next_cursor")
        if not cursor:
            break
        time.sleep(1.5)
    return spiele

print("✅ API-Funktionen definiert")

In [ ]:
# ⏳ Daten laden – dauert ein paar Minuten!
# Tipp: Beim ersten Mal laden, dann als CSV speichern (Zelle weiter unten)

SAISONS = [2022, 2023, 2024, 2025]

alle_spiele = []
for saison in SAISONS:
    print(f"Lade Saison {saison}...")
    spiele = lade_spiele(saison)
    alle_spiele.extend(spiele)
    print(f"  → {len(spiele)} Spiele")

print(f"\n✅ Gesamt: {len(alle_spiele)} Spiele geladen")

In [5]:
# DataFrame erstellen
df = pd.DataFrame([{
    "game_id": g["id"],
    "date": g["date"][:10],
    "home_team": g["home_team"]["full_name"],
    "away_team": g["visitor_team"]["full_name"],
    "home_score": g["home_team_score"],
    "away_score": g["visitor_team_score"],
    "season": g["season"],
    "status": g["status"],
} for g in alle_spiele])

df = df[df["status"] == "Final"].copy()
df["date"] = pd.to_datetime(df["date"])
df["home_win"] = (df["home_score"] > df["away_score"]).astype(int)
df = df.sort_values("date").reset_index(drop=True)

print(f"✅ {len(df)} abgeschlossene Spiele")

NameError: name 'alle_spiele' is not defined

In [6]:
# First, define the alle_spiele variable with your game data
# For example, you might load it from an API or a file
import pandas as pd

# Example of how to define alle_spiele (replace this with your actual data source)
# This could be from an API call like:
# import requests
# response = requests.get("https://your-basketball-api.com/games")
# alle_spiele = response.json()["data"]

# For demonstration, I'll create a simple example
alle_spiele = [
    {
        "id": 1,
        "date": "2023-01-15T19:00:00.000Z",
        "home_team": {"full_name": "Team A"},
        "visitor_team": {"full_name": "Team B"},
        "home_team_score": 105,
        "visitor_team_score": 98,
        "season": 2023,
        "status": "Final"
    },
    # Add more games as needed
]

# DataFrame erstellen
df = pd.DataFrame([{
    "game_id": g["id"],
    "date": g["date"][:10],
    "home_team": g["home_team"]["full_name"],
    "away_team": g["visitor_team"]["full_name"],
    "home_score": g["home_team_score"],
    "away_score": g["visitor_team_score"],
    "season": g["season"],
    "status": g["status"],
} for g in alle_spiele])

df = df[df["status"] == "Final"].copy()
df["date"] = pd.to_datetime(df["date"])
df["home_win"] = (df["home_score"] > df["away_score"]).astype(int)
df = df.sort_values("date").reset_index(drop=True)

print(f"✅ {len(df)} abgeschlossene Spiele")

✅ 1 abgeschlossene Spiele


In [ ]:
# 💾 OPTIONAL: Spieldaten als CSV speichern, damit du nicht jedes Mal laden musst
# Beim nächsten Mal kannst du stattdessen die nächste Zelle nutzen

df.to_csv("spieldaten_cache.csv", index=False)
print("✅ Gespeichert als spieldaten_cache.csv")

In [ ]:
# 💾 OPTIONAL: Spieldaten aus Cache laden (statt API)
# Kommentier diese Zelle ein wenn du die Daten schon mal geladen hast

# df = pd.read_csv("spieldaten_cache.csv", parse_dates=["date"])
# print(f"✅ {len(df)} Spiele aus Cache geladen")

## 3. Daten anschauen
Hier kannst du die Daten erkunden und ein Gefühl dafür bekommen.

In [7]:
# Erste Zeilen anschauen
df.head(10)

,game_id,date,home_team,away_team,home_score,away_score,season,status,home_win
0,1,2023-01-15,Team A,Team B,105,98,2023,Final,1


In [8]:
# Wie viele Spiele pro Saison?
df.groupby("season").size()

season
2023    1
dtype: int64

In [ ]:
# Wie oft gewinnt das Heimteam?
heimsieg_quote = df["home_win"].mean()
print(f"Heimsieg-Quote: {heimsieg_quote:.1%}")
print(f"Das heisst: Ein dummes Modell das immer 'Heim gewinnt' tippt, hätte {heimsieg_quote:.1%} Genauigkeit.")
print(f"Unser Modell muss also besser als {heimsieg_quote:.1%} sein!")

In [ ]:
# Punkteverteilung
print(f"Durchschnitt Heim: {df['home_score'].mean():.1f} Punkte")
print(f"Durchschnitt Auswärts: {df['away_score'].mean():.1f} Punkte")
print(f"Durchschnittlicher Punkteunterschied: {(df['home_score'] - df['away_score']).mean():.1f}")

## 4. Features berechnen
Rolling Winrate und Punkte-Durchschnitte der letzten X Spiele.

**Hier kannst du experimentieren:**
- `FORM_WINDOW` ändern (5, 10, 15, 20)
- Neue Features hinzufügen (z.B. Heim-Winrate nur für Heimspiele)

In [ ]:
def _team_spiele(df, team, vor_datum, window):
    """Letzte X Spiele eines Teams vor einem Datum."""
    mask = (df["date"] < vor_datum) & ((df["home_team"] == team) | (df["away_team"] == team))
    return df.loc[mask].tail(window)


def _winrate(spiele, team):
    """Gewinnrate eines Teams."""
    if len(spiele) == 0:
        return 0.5
    wins = sum(
        (r["home_team"] == team and r["home_win"] == 1) or
        (r["away_team"] == team and r["home_win"] == 0)
        for _, r in spiele.iterrows()
    )
    return wins / len(spiele)


def _avg_pts(spiele, team):
    """Durchschnittlich erzielte und kassierte Punkte."""
    if len(spiele) == 0:
        return 110.0, 110.0
    scored, conceded = [], []
    for _, r in spiele.iterrows():
        if r["home_team"] == team:
            scored.append(r["home_score"])
            conceded.append(r["away_score"])
        else:
            scored.append(r["away_score"])
            conceded.append(r["home_score"])
    return np.mean(scored), np.mean(conceded)


print("✅ Hilfsfunktionen definiert")

In [ ]:
# ⏳ Features berechnen – dauert etwas bei vielen Spielen

print(f"Berechne Features (Window={FORM_WINDOW})...")

records = []
for idx, row in df.iterrows():
    datum = row["date"]
    heim = row["home_team"]
    ausw = row["away_team"]
    
    h_spiele = _team_spiele(df, heim, datum, FORM_WINDOW)
    a_spiele = _team_spiele(df, ausw, datum, FORM_WINDOW)
    
    h_scored, h_conceded = _avg_pts(h_spiele, heim)
    a_scored, a_conceded = _avg_pts(a_spiele, ausw)
    
    records.append({
        "home_winrate": _winrate(h_spiele, heim),
        "away_winrate": _winrate(a_spiele, ausw),
        "home_pts_scored": h_scored,
        "home_pts_conceded": h_conceded,
        "away_pts_scored": a_scored,
        "away_pts_conceded": a_conceded,
    })
    
    # Fortschritt anzeigen
    if idx % 500 == 0:
        print(f"  {idx}/{len(df)} Spiele...")

feat_df = pd.DataFrame(records, index=df.index)
df = pd.concat([df, feat_df], axis=1)

print(f"\n✅ Features fertig!")

In [ ]:
# Features anschauen
df[["home_team", "away_team", "home_win", "home_winrate", "away_winrate", 
    "home_pts_scored", "away_pts_scored"]].tail(10)

## 5. Elo-Rating
Jedes Team bekommt einen Elo-Wert. Gewinnt ein schwaches Team gegen ein starkes, steigt sein Elo mehr.

**Hier kannst du experimentieren:**
- `ELO_K`: Höher = schnellere Reaktion, Niedriger = stabiler
- `ELO_SEASON_REGRESSION`: Wie stark wird Elo zwischen Saisons zurückgesetzt

In [ ]:
def berechne_elo(df, start_elo=ELO_START, k=ELO_K, regression=ELO_SEASON_REGRESSION):
    """Elo-Ratings berechnen mit Season-Regression."""
    teams = pd.concat([df["home_team"], df["away_team"]]).unique()
    elo = {team: start_elo for team in teams}
    
    home_elo_list, away_elo_list = [], []
    prev_season = None
    
    for _, row in df.iterrows():
        # Season Regression
        if prev_season is not None and row["season"] != prev_season:
            for team in elo:
                elo[team] = elo[team] + regression * (start_elo - elo[team])
            print(f"  Elo-Regression: Saison {prev_season} → {row['season']}")
        prev_season = row["season"]
        
        heim = row["home_team"]
        ausw = row["away_team"]
        h_elo = elo[heim]
        a_elo = elo[ausw]
        
        home_elo_list.append(h_elo)
        away_elo_list.append(a_elo)
        
        expected_h = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))
        actual_h = row["home_win"]
        elo[heim] += k * (actual_h - expected_h)
        elo[ausw] += k * ((1 - actual_h) - (1 - expected_h))
    
    df = df.copy()
    df["home_elo"] = home_elo_list
    df["away_elo"] = away_elo_list
    
    return df, elo


df, elo = berechne_elo(df)
print(f"\n✅ Elo fertig!")

In [ ]:
# 🏆 Top 10 Teams nach aktuellem Elo
elo_ranking = sorted(elo.items(), key=lambda x: x[1], reverse=True)
print("\n🏆 ELO RANKING")
print("=" * 40)
for i, (team, rating) in enumerate(elo_ranking[:10], 1):
    print(f"{i:2}. {team:<30} {rating:.0f}")

## 6. Modell trainieren & testen
Hier wird das eigentliche ML-Modell trainiert.

**Hier kannst du am meisten experimentieren:**
- Andere Modelle: `RandomForestClassifier`, `GradientBoostingClassifier`, `XGBClassifier`
- Features hinzufügen/entfernen
- Train/Test Split ändern

In [ ]:
FEATURES = [
    "home_winrate", "away_winrate",
    "home_pts_scored", "home_pts_conceded",
    "away_pts_scored", "away_pts_conceded",
    "home_elo", "away_elo",
]

test_season = 2025
train = df[df["season"] < test_season]
test = df[df["season"] == test_season]

print(f"Training: {len(train)} Spiele (Saisons vor {test_season})")
print(f"Test:     {len(test)} Spiele (Saison {test_season})")

In [ ]:
# Logistic Regression (Basis-Modell)
model = LogisticRegression(max_iter=1000)
model.fit(train[FEATURES], train["home_win"])

# Genauigkeit auf Testdaten
test_pred = model.predict(test[FEATURES])
acc = accuracy_score(test["home_win"], test_pred)

print(f"\n✅ Modell-Genauigkeit: {acc:.1%}")
print(f"   (Zum Vergleich: Immer Heim tippen = {test['home_win'].mean():.1%})")

In [ ]:
# Feature Importance – welche Features sind am wichtigsten?
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Gewicht": model.coef_[0]
}).sort_values("Gewicht", ascending=False)

print("\n📊 FEATURE GEWICHTE")
print("=" * 40)
for _, row in importance.iterrows():
    balken = "█" * int(abs(row['Gewicht']) * 10)
    vorzeichen = "+" if row['Gewicht'] > 0 else "-"
    print(f"{row['Feature']:<25} {vorzeichen}{abs(row['Gewicht']):.3f} {balken}")

## 7. Heutige Spiele vorhersagen
Basis-Vorhersagen ohne Verletzungskorrektur.

In [ ]:
# Datum festlegen – ändere hier das Datum wenn du willst
DATUM = datetime.now().strftime("%Y-%m-%d")
# DATUM = "2026-03-17"  # oder ein bestimmtes Datum

print(f"Lade Spiele für: {DATUM}")
data = api_request("games", {"dates[]": DATUM, "per_page": 100})

if data:
    heute_spiele = data["data"]
    print(f"\n✅ {len(heute_spiele)} Spiele gefunden:")
    for s in heute_spiele:
        print(f"  {s['home_team']['full_name']} vs {s['visitor_team']['full_name']}")
else:
    heute_spiele = []
    print("❌ Keine Spiele gefunden")

In [ ]:
# Vorhersagen erstellen
vorhersagen = []

for spiel in heute_spiele:
    heim = spiel["home_team"]["full_name"]
    ausw = spiel["visitor_team"]["full_name"]
    now = pd.Timestamp.now()
    
    h_spiele = _team_spiele(df, heim, now, FORM_WINDOW)
    a_spiele = _team_spiele(df, ausw, now, FORM_WINDOW)
    
    if len(h_spiele) == 0 or len(a_spiele) == 0:
        print(f"⚠️ Keine Daten für {heim} vs {ausw}")
        continue
    
    x = pd.DataFrame([{
        "home_winrate": _winrate(h_spiele, heim),
        "away_winrate": _winrate(a_spiele, ausw),
        "home_pts_scored": _avg_pts(h_spiele, heim)[0],
        "home_pts_conceded": _avg_pts(h_spiele, heim)[1],
        "away_pts_scored": _avg_pts(a_spiele, ausw)[0],
        "away_pts_conceded": _avg_pts(a_spiele, ausw)[1],
        "home_elo": elo.get(heim, ELO_START),
        "away_elo": elo.get(ausw, ELO_START),
    }])
    
    prob = model.predict_proba(x)[0][1]
    gewinner = heim if prob > 0.5 else ausw
    konfidenz = prob if prob > 0.5 else 1 - prob
    
    vorhersagen.append({
        "Heimteam": heim,
        "Auswärtsteam": ausw,
        "Heimsieg %": round(prob * 100, 1),
        "Tipp": gewinner,
        "Konfidenz": round(konfidenz * 100, 1),
    })

vorhersagen_df = pd.DataFrame(vorhersagen)

print("\n📊 BASIS-VORHERSAGEN")
print("=" * 70)
if len(vorhersagen_df) > 0:
    display(vorhersagen_df)
else:
    print("Keine Vorhersagen möglich")

## 8. Verletzungen & Impact
Verletzungsdaten von ESPN holen und Impact Score aus der lokalen CSV lesen.

In [ ]:
# Verletzungen von ESPN laden
try:
    r = requests.get("https://www.espn.com/nba/injuries", 
                     headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")
    
    verletzungen = []
    tabellen = soup.find_all("div", class_="ResponsiveTable")
    for tabelle in tabellen:
        team_span = tabelle.find("span", class_="injuries__teamName")
        espn_name = team_span.text.strip() if team_span else "Unbekannt"
        full_name = TEAM_NAME_MAP.get(espn_name, espn_name)
        
        rows = tabelle.find_all("tr")[1:]
        for row in rows:
            cols = row.find_all("td")
            if len(cols) >= 4:
                verletzungen.append({
                    "Team": full_name,
                    "Spieler": cols[0].text.strip(),
                    "Status": cols[3].text.strip(),
                })
    
    verletzungen_df = pd.DataFrame(verletzungen)
    verletzungen_df = verletzungen_df[verletzungen_df["Status"].str.contains("Out|Doubtful", case=False, na=False)]
    print(f"✅ {len(verletzungen_df)} verletzte/fragliche Spieler")
except Exception as e:
    print(f"❌ Fehler: {e}")
    verletzungen_df = pd.DataFrame(columns=["Team", "Spieler", "Status"])

In [ ]:
# Verletzte Spieler anschauen
if len(verletzungen_df) > 0:
    display(verletzungen_df.head(20))
else:
    print("Keine Verletzungsdaten")

In [ ]:
# Impact Scores laden (aus player_stats.csv)
try:
    impact_df = pd.read_csv("player_stats.csv")
    print(f"✅ {len(impact_df)} Spieler mit Impact Score geladen")
    
    # Top 10 anzeigen
    print("\n🌟 TOP 10 IMPACT SCORES:")
    display(impact_df.nlargest(10, "Impact_Final")[["PLAYER_NAME", "TEAM_ABBREVIATION", "PTS", "AST", "REB", "Impact_Final"]])
except FileNotFoundError:
    print("❌ player_stats.csv nicht gefunden!")
    print("   Führe zuerst 'python3 update_stats.py' im Terminal aus.")
    impact_df = pd.DataFrame()

## 9. Finale Vorhersagen (mit Verletzungskorrektur)

In [3]:
def berechne_impact_verlust(team_name, verletzungen_df, impact_df):
    """Berechnet wie viel Impact ein Team durch Verletzungen verliert."""
    if len(verletzungen_df) == 0 or len(impact_df) == 0:
        return 0.0, []
    
    verletzt = verletzungen_df[verletzungen_df["Team"] == team_name]
    total_impact = 0.0
    details = []
    
    for _, spieler in verletzt.iterrows():
        name = spieler["Spieler"]
        status = spieler["Status"]
        teile = name.split()
        
        if len(teile) >= 2:
            match = impact_df[
                impact_df["PLAYER_NAME"].str.contains(teile[0], case=False, na=False) &
                impact_df["PLAYER_NAME"].str.contains(teile[-1], case=False, na=False)
            ]
        else:
            match = impact_df[impact_df["PLAYER_NAME"].str.contains(name, case=False, na=False)]
        
        if len(match) > 0:
            impact = match.iloc[0]["Impact_Final"]
            gewicht = 1.0 if "Out" in status else 0.5
            total_impact += impact * gewicht
            details.append(f"{name} ({impact:.1f}, {status})")
    
    return total_impact, details

print("✅ Funktion definiert")

✅ Funktion definiert


---
## 10. 🧪 Experimentier-Bereich
Ab hier kannst du frei rumspielen! Hier ein paar Ideen:

In [ ]:
# 🧪 EXPERIMENT: Anderes Modell ausprobieren
# Entkommentiere eine der folgenden Zeilen:

# from sklearn.ensemble import RandomForestClassifier
# model_exp = RandomForestClassifier(n_estimators=100, random_state=42)

# from sklearn.ensemble import GradientBoostingClassifier
# model_exp = GradientBoostingClassifier(n_estimators=100, random_state=42)

# model_exp.fit(train[FEATURES], train["home_win"])
# acc_exp = accuracy_score(test["home_win"], model_exp.predict(test[FEATURES]))
# print(f"Genauigkeit neues Modell: {acc_exp:.1%}")
# print(f"Genauigkeit altes Modell: {acc:.1%}")

In [ ]:
# 🧪 EXPERIMENT: Verschiedene FORM_WINDOW Werte testen

# for window in [5, 10, 15, 20]:
#     # Hier müsstest du berechne_features nochmal laufen lassen
#     # mit dem jeweiligen Window – das dauert aber lange.
#     # Tipp: Teste erstmal nur mit einer Saison!
#     pass

In [ ]:
# 🧪 EXPERIMENT: Eigene Features hinzufügen
# Ideen:
# - Ruhetage seit dem letzten Spiel
# - Head-to-Head Bilanz (wie oft hat Team A gegen Team B gewonnen?)
# - Heim-Winrate nur für Heimspiele (nicht alle Spiele)
# - Punktedifferenz (scored - conceded) als einzelnes Feature

# Beispiel: Punktedifferenz
# df["home_pts_diff"] = df["home_pts_scored"] - df["home_pts_conceded"]
# df["away_pts_diff"] = df["away_pts_scored"] - df["away_pts_conceded"]